# HW1. Решение

ноутбук только грузит модели и пишет csv. обучение тем в `task2_topics.ipynb`.

пути в первой ячейке кода. на проверке их подставляют через окружение.

In [1]:
import os

import pandas as pd

from solution_utils import (
    compute_metrics,
    compute_partition_metrics,
    init_reaction_models,
    init_topic_models,
    predict_reaction,
    predict_topic,
    read_dialogs,
    read_labels,
    save_predictions,
)

In [2]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

# локально jsonl лежит в data/, на проверке пути приходят из окружения
if not os.path.exists(TRAIN_PATH) and os.path.exists("data/hw1_train.jsonl"):
    TRAIN_PATH = "data/hw1_train.jsonl"
if not os.path.exists(DATA_PATH) and os.path.exists("data/hw1_train.jsonl"):
    DATA_PATH = "data/hw1_train.jsonl"

dialogs = read_dialogs(DATA_PATH)
print("данные:", DATA_PATH)
print("диалогов:", len(dialogs))

данные: data/hw1_train.jsonl
диалогов: 10000


## Задача 1. Реакция пользователя

In [3]:
reaction_models = init_reaction_models(TRAIN_PATH)
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

reaction_predictions, reaction_metrics = {}, {}
for name, model in reaction_models.items():
    reaction_predictions[name] = predict_reaction(model, dialogs)
    if reaction_labels:
        reaction_metrics[name] = compute_metrics(dialogs, reaction_predictions[name], reaction_labels)

pd.DataFrame(reaction_metrics).T.round(3)

,accuracy
dummy_most_frequent,0.591
dummy_stratified,0.455


## Задача 2. Темы

модели из `artifacts/` ставят имя темы. на скрытом тесте
сравнивают разбиения, не сами названия. меток тем в обучающем
файле нет, таблица ниже будет пустой.

In [4]:
topic_models = init_topic_models(TRAIN_PATH)
topic_labels = read_labels(LABELS_PATH, "topic") if LABELS_PATH else {}

topic_predictions, topic_metrics = {}, {}
for name, model in topic_models.items():
    topic_predictions[name] = predict_topic(model, dialogs)
    if topic_labels:
        topic_metrics[name] = compute_partition_metrics(dialogs, topic_predictions[name], topic_labels)

pd.DataFrame(topic_metrics).T.round(3)

""


## Ответы всех моделей

In [5]:
save_predictions(
    PREDICTIONS_PATH,
    dialogs,
    {"reaction": reaction_predictions, "topic": topic_predictions},
)
n_models = len(reaction_predictions) + len(topic_predictions)
print("строк в файле ответов:", len(dialogs) * n_models)

строк в файле ответов: 60000
